Definições de funções.

In [1]:
# Importar bibliotecas necessárias
import pandas as pd
import re

# Função para remover a parte numérica inicial e os possíveis sublinhado que a segue.
def clean_category(category):
  # Dicionário de correções de termos jurídicos.
  corrections_dict = {
      'tributario': 'tributário'
  }
  cleaned_numeric_part = re.sub(r'^\d+_?', '', category)

  # Substituir todos os sublinhados por espaços
  final_string = cleaned_numeric_part.replace('_', ' ')

  # Aplicar correções.
  words = final_string.split()
  corrected_words = []
  for word in words:
    # Verifica se a palavra está no dicionário de correções e aplica
    corrected_words.append(corrections_dict.get(word, word))
  final_string = ' '.join(corrected_words)

  # Converter para modo título: primeira letra de cada palavra maiúscula, as demais minúsculas,
  # com as exceções de alguns artigos e preosições.
  lowercase_exceptions = {'e', 'o', 'a', 'de', 'do', 'dos', 'da', 'das'}
  processed_words = []
  for word in final_string.split():
    if word.lower() in lowercase_exceptions:
      processed_words.append(word.lower())
    else:
      processed_words.append(word.title())
  return ' '.join(processed_words)

# Função que recebe a url e retorna um Dataframe.
def load_dataframe(url):
   return pd.read_json(url, lines=True)

# Função recuperar todas as categorias de um Dataframe de forma distinta.
def get_unique_categories(df):
  return df['category'].unique()

# Função para juntar dois Dataframes por um campo chave e opcionalmente selecionar colunas.
def merge_dataframes(df1, df2, key, columns=None):
  merged_df = pd.merge(df1, df2, on=key, how='inner')
  if columns is not None:
    return merged_df[columns]
  return merged_df

Questões disponíveis no github.

In [2]:
# URL das questões.
url_questions = "https://raw.githubusercontent.com/maritaca-ai/oab-bench/main/data/oab_bench/question.jsonl"

# Carregar um Dataframe com as perguntas a partir do arquivo .jsonl.
df_questions = load_dataframe(url_questions)

# Inserir uma coluna para enumerar as questões, com contagem a partir do número 1.
df_questions.insert(0, 'question', range(1, len(df_questions)+1))


Respostas de referências às questões anteriores e disponíveis no mesmo repositório do github.

In [3]:
# URL das respostas de referência.
url_guidelines = "https://raw.githubusercontent.com/maritaca-ai/oab-bench/refs/heads/main/data/oab_bench/reference_answer/guidelines.jsonl"

# Carregar um Dataframe com as respostas a partir do arquivo .jsonl.
df_guidelines = load_dataframe(url_guidelines)

Dataframes de questões e de referências do intervalo de um integrante da equipe.

In [4]:
# Povoar subconjutos tanto das perguntas, quanto das respostas.
# Subconjunto das questões de 31 a 40.
# O índice do Dataframe começa do 0 (zero), portanto, a linha 31 para meu exercício é a 30 no pandas.
#   analogamente a 40 p é a 39 no pandas.
# iloc selciona um intervalo fechado à esquerda e aberto à direita, logo pegar o intervalo de 31 a 40 para mim
#   é no iloc de 30 (inclusive) até 40 (exclusive).

# Meu sub-grupo de questões e respostas.
x = 30
y = 40
df_my_questions = df_questions.iloc[x:y].copy()
df_my_guidelines = df_guidelines.iloc[x:y].copy()

Limpeza de dados em desuso em memória.

In [5]:
# Excluir Dataframes que não são mais necessários.
import gc

del df_questions
del df_guidelines
gc.collect()

0

Diferentes categorias envolvendo as questões de um integrante da equipe.

In [6]:
# Realizar limpeza no campo das categorias.
df_my_questions['category'] = df_my_questions['category'].apply(clean_category)

Junção dos Dataframes de questoes e respostas de referências colocadas lado a lado em um novo e único Dataframe.

In [9]:
# Junção dos DataFrame df_my_question e df_my_guidelines em um único DataFrame usando o question_id de ambos como atributo da interseção.
# Neste Dataframe dá para ver a pergunta e a resposta da linha guia juntas.
# E, também escolher as colunas que serão apresentadas.
key = 'question_id'
columns=['question', 'question_id', 'category', 'statement', 'turns', 'choices']
df_questions_and_guidelines = merge_dataframes(df_my_questions, df_my_guidelines, key, columns)

# Exibir o Dataframe completo, que são 10 linhas.
display(df_questions_and_guidelines)


,question,question_id,category,statement,turns,choices
0,31,39_direito_tributario_peca_profissional,Direito Tributário,"PEÇA PRÁTICO-PROFISSIONAL\n\nPedro de Camões, ...",[],"[{'index': 0, 'turns': ['A medida judicial cab..."
1,32,39_direito_tributario_questao_1,Direito Tributário,QUESTÃO\n\nA sociedade empresária Metalúrgica ...,"[A partir de quando se deve contar, no caso co...","[{'index': 0, 'turns': ['O prazo para oferta d..."
2,33,39_direito_tributario_questao_2,Direito Tributário,QUESTÃO\n\nA sociedade empresária Tudo Verde L...,[A qual dos municípios o ISS de jardinagem é d...,"[{'index': 0, 'turns': ['O ISS de jardinagem é..."
3,34,39_direito_tributario_questao_3,Direito Tributário,QUESTÃO\n\nA Administração Tributária Federal ...,[É válida a exigência de depósito do montante ...,"[{'index': 0, 'turns': ['Não, pois é inconstit..."
4,35,39_direito_tributario_questao_4,Direito Tributário,QUESTÃO\n\nA sociedade empresária Faz Tudo Ltd...,[Está correto o argumento da necessidade de no...,"[{'index': 0, 'turns': ['Não está correto, por..."
5,36,40_direito_administrativo_peca_profissional,Direito Administrativo,PEÇA PRÁTICO-PROFISSIONAL\n\nO Ministério Públ...,[],"[{'index': 0, 'turns': ['O(a) examinando(a) de..."
6,37,40_direito_administrativo_questao_1,Direito Administrativo,QUESTÃO\n\nA sociedade empresária Sagaz S.A. e...,[Há necessidade de demonstração do elemento su...,"[{'index': 0, 'turns': ['Não. A responsabiliza..."
7,38,40_direito_administrativo_questao_2,Direito Administrativo,QUESTÃO\n\nDeterminada informação de interesse...,[É lícita a cobrança efetuada pelo órgão respo...,"[{'index': 0, 'turns': ['Não. A submissão e o ..."
8,39,40_direito_administrativo_questao_3,Direito Administrativo,QUESTÃO\n\nCerta Secretaria do Estado Alfa fez...,[É possível a utilização do sistema de registr...,"[{'index': 0, 'turns': ['Sim. A Administração ..."
9,40,40_direito_administrativo_questao_4,Direito Administrativo,"QUESTÃO\n\nRecentemente, Iná foi aprovada em c...",[A aprovação de Iná no mencionado concurso imp...,"[{'index': 0, 'turns': ['Não. Iná foi aprovada..."
